# Clase 217 — Matrix factorization: SVD truncado + ALS implicit

Requiere: `pip install scipy scikit-learn implicit`. Sobre dataset sintético similar a MovieLens 100K.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

rng = np.random.default_rng(42)
n_users, n_items, K_TRUE = 1000, 500, 8

# Generamos datos con estructura latente conocida (k=8) para validar que MF recupera estructura
P_true = rng.normal(0, 1, (n_users, K_TRUE))
Q_true = rng.normal(0, 1, (n_items, K_TRUE))
R_full = P_true @ Q_true.T   # rating ideal sin ruido

# Solo observamos 8% de las interacciones (sparse)
mask = rng.random((n_users, n_items)) < 0.08
R_obs = np.where(mask, np.clip(R_full + rng.normal(0, 0.3, R_full.shape), -3, 3), 0)
print(f'observed: {mask.sum():,} / {n_users * n_items:,} ({mask.mean():.2%})')

## 1. SVD truncado sobre matriz imputada con 0

In [ ]:
R_sparse = csr_matrix(R_obs)
U, sigma, Vt = svds(R_sparse, k=K_TRUE)
# svds devuelve sigma en orden ascendente — invertimos
order = np.argsort(-sigma)
U, sigma, Vt = U[:, order], sigma[order], Vt[order]
print('sigma (top-8):', np.round(sigma, 2))

R_hat = U @ np.diag(sigma) @ Vt
# Error sobre observed
rmse_obs = np.sqrt(((R_obs[mask] - R_hat[mask]) ** 2).mean())
# Error sobre held-out (lo que SVD "completó")
rmse_unseen = np.sqrt(((R_full[~mask] - R_hat[~mask]) ** 2).mean())
print(f'RMSE sobre observed:  {rmse_obs:.3f}')
print(f'RMSE sobre held-out:  {rmse_unseen:.3f} (SVD imputed → muy ruidoso porque imputamos 0)')

## 2. ALS "a mano" — solo training observed

In [ ]:
def als(R_obs, mask, k=8, lam=0.05, n_iter=20):
    """ALS explicit: alterna actualizar P y Q solo en los entries observed."""
    n_u, n_i = R_obs.shape
    P = np.random.default_rng(0).normal(0, 0.1, (n_u, k))
    Q = np.random.default_rng(1).normal(0, 0.1, (n_i, k))

    for it in range(n_iter):
        # Actualizar P fijando Q
        for u in range(n_u):
            items_u = np.where(mask[u])[0]
            if len(items_u) == 0: continue
            Q_u = Q[items_u]
            A = Q_u.T @ Q_u + lam * np.eye(k)
            b = Q_u.T @ R_obs[u, items_u]
            P[u] = np.linalg.solve(A, b)
        # Actualizar Q fijando P
        for i in range(n_i):
            users_i = np.where(mask[:, i])[0]
            if len(users_i) == 0: continue
            P_i = P[users_i]
            A = P_i.T @ P_i + lam * np.eye(k)
            b = P_i.T @ R_obs[users_i, i]
            Q[i] = np.linalg.solve(A, b)

    return P, Q

P, Q = als(R_obs, mask, k=K_TRUE, lam=0.05, n_iter=15)
R_hat_als = P @ Q.T
rmse_obs = np.sqrt(((R_obs[mask] - R_hat_als[mask]) ** 2).mean())
rmse_unseen = np.sqrt(((R_full[~mask] - R_hat_als[~mask]) ** 2).mean())
print(f'ALS RMSE observed:    {rmse_obs:.3f}')
print(f'ALS RMSE held-out:    {rmse_unseen:.3f}  (mucho mejor que SVD imputado con 0)')

## 3. Implicit ALS con la lib `implicit` (Hu et al. 2008)

In [ ]:
# Convertimos a implicit: rating > 1 = preferencia
R_implicit = csr_matrix(np.where(R_obs > 1, R_obs, 0))
print(f'implicit interactions: {R_implicit.nnz:,}')

try:
    import os; os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')   # avoid threading warning
    import implicit
    model = implicit.als.AlternatingLeastSquares(
        factors=32, regularization=0.05, iterations=15, alpha=40, random_state=42,
    )
    model.fit(R_implicit, show_progress=False)
    print('item_factors shape:', model.item_factors.shape)
    print('user_factors shape:', model.user_factors.shape)

    # Recomendar para user_id=42
    user_id = 42
    ids, scores = model.recommend(user_id, R_implicit[user_id], N=10)
    print(f'\ntop-10 items para user_{user_id}:')
    for i, s in zip(ids, scores):
        print(f'  item_{i}: {s:.3f}')
except ImportError:
    print('pip install implicit para esta celda')

## 4. Inspeccionar embeddings — items similares

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Top 5 items más similares al item_id=10 según los embeddings ALS
target_item = 10
sims = cosine_similarity(Q[target_item:target_item + 1], Q).ravel()
sims[target_item] = -1
top = np.argsort(-sims)[:5]
print(f'items más similares a item_{target_item} según embeddings ALS:')
for i in top:
    print(f'  item_{i}: sim={sims[i]:.4f}')

# Comparar con ground truth (las que comparten factores latentes verdaderos)
sims_true = cosine_similarity(Q_true[target_item:target_item + 1], Q_true).ravel()
sims_true[target_item] = -1
top_true = np.argsort(-sims_true)[:5]
overlap = set(top) & set(top_true)
print(f'\noverlap con top-5 según embeddings verdaderos: {len(overlap)}/5')

## 5. PCA 2D de item embeddings

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Sintetizamos 'género' como cluster en el espacio latente verdadero
genres = np.argmax(np.abs(Q_true[:, :4]), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, X, title in [(axes[0], Q_true, 'true latent'), (axes[1], Q, 'ALS recovered')]:
    coords = PCA(n_components=2, random_state=0).fit_transform(X)
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=genres, cmap='tab10', alpha=0.6, s=20)
    ax.set_title(title)
    plt.colorbar(sc, ax=ax, label='"genre"')
plt.tight_layout()
plt.savefig('embeddings_pca.png', dpi=80)
plt.show()
print('→ ALS recuperó la estructura latente: "géneros" forman clusters separables.')

## Ejercicio guiado

1. Bajá MovieLens 1M real. Entrená ALS implicit con `factors=64`, `alpha=40`, `iterations=20`. Mostrá top-10 películas para un usuario, con título y género.
2. PCA 2D de `model.item_factors`, colorear por género real. ¿Se separan?
3. Cross-validation con `factors ∈ {20, 50, 100}` × `regularization ∈ {0.001, 0.01, 0.1}`. Reportar mejor combo por NDCG@10.
4. Implementar fallback para cold-start: usuario nuevo → top-N por popularidad (`b_i`) hasta tener 5 interacciones.
5. Bonus: comparar tiempo de entrenamiento `implicit.als` vs `pyspark.ml.recommendation.ALS` en cluster (Clase 210).

## Conclusiones

- Matrix factorization aprende **embeddings k-dim** que capturan factores latentes (género, gusto).
- ALS sobre matriz observed es ~10× mejor que SVD imputado con 0.
- Implicit ALS (Hu et al.) es el caballo de batalla: rápido, escalable, robusto.
- Embeddings sirven mucho más allá de top-N: similar items, clustering, two-tower retrieval con FAISS.